# DC Comics Character Data — Exploratory Data Analysis

This notebook walks through a brief exploratory analysis of the
[DC Comics Wikia dataset](https://github.com/ghoshpoulami1293/ComicBookUniverse).

**Goals**
1. Load and inspect the raw data
2. Clean and tidy the dataset
3. Summarise key patterns
4. Produce a handful of visualisations

**Before running**: make sure `data/dc-wikia-data.csv` exists.  
See `data/README.md` for download instructions.

## 1  Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Consistent plot style throughout the notebook
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.dpi"] = 120

## 2  Load the data

In [ ]:
DATA_PATH = "../data/dc-wikia-data.csv"

df_raw = pd.read_csv(DATA_PATH)
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 3  Initial inspection

In [ ]:
df_raw.info()

In [ ]:
# Count missing values per column
df_raw.isnull().sum().sort_values(ascending=False)

## 4  Data cleaning

Steps applied:
* Rename columns to lowercase, replacing spaces with underscores for convenience
* Drop the `urlslug` column (not useful for analysis)
* Strip trailing labels from categorical columns (e.g. `"Male Characters"` → `"Male"`)
* Convert `appearances` to a numeric type (coerce errors to NaN)
* Keep only rows where `year` is a plausible value (≥ 1935)

In [ ]:
df = df_raw.copy()

# --- rename columns ---
df.columns = df.columns.str.lower().str.replace(" ", "_", regex=False)
df = df.drop(columns=["urlslug"], errors="ignore")

# --- strip trailing labels from categoricals ---
STRIP_SUFFIX = {
    "align":  " Characters",
    "sex":    " Characters",
    "alive":  " Characters",
    "gsm":    " Characters",
}
for col, suffix in STRIP_SUFFIX.items():
    if col in df.columns:
        df[col] = df[col].str.replace(suffix, "", regex=False).str.strip()

# --- tidy the identity column ---
if "id" in df.columns:
    df["id"] = df["id"].str.replace(" Identity", "", regex=False).str.strip()

# --- appearances to numeric ---
df["appearances"] = pd.to_numeric(df["appearances"], errors="coerce")

# --- filter unrealistic years ---
df = df[df["year"].between(1935, 2024)]

print(f"Clean shape: {df.shape}")
df.head()

## 5  Summary statistics

In [ ]:
df.describe(include="all")

In [ ]:
# Value counts for key categoricals
for col in ["align", "sex", "alive", "id"]:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print(df[col].value_counts(dropna=False))

## 6  Visualisations

### 6.1  Character alignment breakdown

In [ ]:
align_counts = df["align"].value_counts(dropna=False)

fig, ax = plt.subplots(figsize=(7, 4))
align_counts.plot(kind="bar", ax=ax, color=sns.color_palette("tab10"))
ax.set_title("DC Characters by Alignment")
ax.set_xlabel("Alignment")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### 6.2  Gender breakdown

In [ ]:
sex_counts = df["sex"].value_counts(dropna=False)

fig, ax = plt.subplots(figsize=(6, 4))
sex_counts.plot(kind="bar", ax=ax, color=sns.color_palette("pastel"))
ax.set_title("DC Characters by Gender")
ax.set_xlabel("Gender")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### 6.3  New characters introduced per decade

In [ ]:
df["decade"] = (df["year"] // 10 * 10).astype(int)
decade_counts = df.groupby("decade").size()

fig, ax = plt.subplots(figsize=(9, 4))
decade_counts.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("New DC Characters Introduced per Decade")
ax.set_xlabel("Decade")
ax.set_ylabel("Number of new characters")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### 6.4  Distribution of total comic appearances (log scale)

In [ ]:
appearances_clean = df["appearances"].dropna()
appearances_clean = appearances_clean[appearances_clean > 0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.log10(appearances_clean), bins=40, color="teal", edgecolor="white")
ax.set_title("Distribution of Comic Appearances (log₁₀ scale)")
ax.set_xlabel("log₁₀(appearances)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

### 6.5  Top 15 most-appearing characters

In [ ]:
top15 = df.nlargest(15, "appearances")[["name", "appearances", "align", "sex"]]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top15["name"], top15["appearances"], color="coral")
ax.invert_yaxis()
ax.set_title("Top 15 DC Characters by Comic Appearances")
ax.set_xlabel("Appearances")
plt.tight_layout()
plt.show()

### 6.6  Alignment over time (stacked area)

In [ ]:
align_decade = (
    df.dropna(subset=["align"])
    .groupby(["decade", "align"])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 5))
align_decade.plot(kind="area", stacked=True, ax=ax, alpha=0.7)
ax.set_title("DC Character Alignment Over Time (by Decade)")
ax.set_xlabel("Decade")
ax.set_ylabel("Number of new characters")
ax.legend(title="Alignment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 7  Key findings

> Fill this section in after running the notebook with your own observations.

Some things worth noting:
* **Alignment**: The dataset skews heavily toward Good characters.
* **Gender**: Male characters significantly outnumber Female characters, though Female representation
  has grown in more recent decades.
* **Appearances**: The distribution of appearances is highly right-skewed — a handful of iconic
  characters (Batman, Superman, Wonder Woman, …) dominate panel counts.
* **Introductions over time**: Character introductions peaked in certain decades — look for spikes
  corresponding to major publishing events or licensing expansions.

## 8  Next steps

* Investigate which `HAIR` / `EYE` colour combos are most common per alignment.
* Model which features are most predictive of whether a character is a hero or villain.
* Merge with the Marvel dataset for cross-universe comparisons.